# Amazon Book Price Estimator
## `Run this notebook before book_pricer_models.ipynb`

This is a machine learning project that fine-tunes a large language model to predict 
the price of a book from its title, author, genre and format alone.

## Data Preparation

The dataset used in this project is the **Amazon Books Dataset** from Kaggle.

- Source: Kaggle
- Author: Chhavi Dhankhar  
- Link: https://www.kaggle.com/datasets/chhavidhankhar11/amazon-books-dataset

This dataset contains information on **7,167 Amazon books**, including:

- Title
- Author
- Genre
- Sub-Genre
- Format
- Price (USD)

The dataset was cleaned and exported as `books_clean.csv` before modelling.

In [1]:
!pip install gdown -q   # install gdown to download the csv database file from Google Drive


zsh:1: command not found: pip


In [2]:
import pandas as pd
import numpy as np
import pickle
import random
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Optional
import gdown

print('Ready')

Ready


## Item Class

Each book becomes an Item with three fields:
- `item.title` — book title (used for chart labels in evaluate())
- `item.price` — actual price in USD (ground truth)
- `item.prompt` — text the LLM sees to make its prediction

In [3]:
@dataclass
class Item:
    title: str
    price: float
    prompt: str

    def __repr__(self):
        return f"Item(price=${self.price:.2f}, title='{self.title[:50]}')"


# Quick contract check
sample = Item(
    title="A Brief History of Time",
    price=3.75,
    prompt="How much does this book cost?\n\nTitle: A Brief History of Time\nAuthor: Stephen Hawking\nGenre: Science\nSub Genre: Physics\nFormat: Paperback"
)
print(repr(sample))
print()
print(sample.prompt)

Item(price=$3.75, title='A Brief History of Time')

How much does this book cost?

Title: A Brief History of Time
Author: Stephen Hawking
Genre: Science
Sub Genre: Physics
Format: Paperback


## Load Dataset

In [ ]:
# Download books_clean.csv from Google Drive
file_id = '1f8wj-y-zddemezdybk8DkXJEyOp5OgJC'  #  file ID on Google Drive
gdown.download(f'https://drive.google.com/uc?id={file_id}', 'books_clean.csv', quiet=False)

df = pd.read_csv('books_clean.csv')
print(f'Loaded {len(df):,} books')
print(f'Columns: {list(df.columns)}')
df.head(3)

Downloading...
From: https://drive.google.com/uc?id=1f8wj-y-zddemezdybk8DkXJEyOp5OgJC
To: /Users/user/projects/llm_engineering/week6/community-contributions/geraldino/books_clean.csv
100%|██████████| 1.00M/1.00M [00:00<00:00, 1.48MB/s]

Loaded 7,167 books
Columns: ['Title', 'Author', 'Main Genre', 'Sub Genre', 'Type', 'Price']


,Title,Author,Main Genre,Sub Genre,Type,Price
0,The Complete Novel of Sherlock Holmes,Arthur Conan Doyle,"Arts, Film & Photography",Cinema & Broadcast,Paperback,2.01
1,Black Holes (L) : The Reith Lectures [Paperbac...,Stephen Hawking,"Arts, Film & Photography",Cinema & Broadcast,Paperback,1.18
2,The Kite Runner,Khaled Hosseini,"Arts, Film & Photography",Cinema & Broadcast,Kindle Edition,2.09


## Build Items

Each row becomes a prompt combining title, author, genre, sub genre and format.
This is the only text the LLM will see when predicting price.

In [ ]:
def build_prompt(row) -> str:
    return (
        f"How much does this book cost?\n\n"
        f"Title: {str(row['Title']).strip()}\n"
        f"Author: {str(row['Author']).strip()}\n"
        f"Genre: {str(row['Main Genre']).strip()}\n"
        f"Sub Genre: {str(row['Sub Genre']).strip()}\n"
        f"Format: {str(row['Type']).strip()}"
    )


def row_to_item(row) -> Optional[Item]:
    try:
        price = float(row['Price'])
        title = str(row['Title']).strip()
        prompt = build_prompt(row)
        return Item(title=title, price=price, prompt=prompt)
    except:
        return None


all_items = [row_to_item(row) for _, row in df.iterrows()]
all_items = [i for i in all_items if i is not None]

prices = [i.price for i in all_items]
print(f'Built {len(all_items):,} items')
print(f'Price range: ${min(prices):.2f} - ${max(prices):.2f}')
print(f'Median:      ${sorted(prices)[len(prices)//2]:.2f}')
print(f'Mean:        ${sum(prices)/len(prices):.2f}')
print()
print('Sample prompt:')
print(all_items[0].prompt)

## Price Signal Analysis

If genre and format predict price, the LLM can learn this from text alone.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

genre_prices = df.groupby('Main Genre')['Price'].mean().sort_values(ascending=False)
genre_prices.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Average Price by Genre')
axes[0].set_xlabel('Price (USD)')

format_prices = df.groupby('Type')['Price'].mean().sort_values(ascending=False)
format_prices.plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Average Price by Format')
axes[1].set_xlabel('Price (USD)')

plt.tight_layout()
plt.show()

print('Top 5 most expensive genres:')
for genre, price in genre_prices.head().items():
    bar = chr(9608) * int(price * 5)
    print(f'  {genre:40s} ${price:.2f}  {bar}')

print()
print('Price by format:')
for fmt, price in format_prices.items():
    bar = chr(9608) * int(price * 5)
    print(f'  {fmt:20s} ${price:.2f}  {bar}')

## Train / Val / Test Split

- **train** — 80%, used to fine-tune the LLM
- **val** — 10%, monitors for overfitting during training
- **test** — 10%, frozen — only ever used inside evaluate()

Fixed seed ensures the split is identical every run.

In [ ]:
random.seed(42)
random.shuffle(all_items)

n = len(all_items)
train = all_items[:int(n * 0.8)]
val   = all_items[int(n * 0.8):int(n * 0.9)]
test  = all_items[int(n * 0.9):]

print(f'Train: {len(train):,}')
print(f'Val:   {len(val):,}')
print(f'Test:  {len(test):,}')
print()

for name, split in [('train', train), ('val', val), ('test', test)]:
    sp = [i.price for i in split]
    print(f'{name:6s}  median=${sorted(sp)[len(sp)//2]:.2f}  mean=${sum(sp)/len(sp):.2f}')

## Save Splits.pkl file on your local device to be used by the book_pricer_models.ipynb


In [ ]:
with open('splits.pkl', 'wb') as f:
    pickle.dump({'train': train, 'val': val, 'test': test}, f)

print(f'Saved splits.pkl')
print(f'  train: {len(train):,} items')
print(f'  val:   {len(val):,} items')
print(f'  test:  {len(test):,} items — FROZEN, only evaluate() touches this')
print()
print('Ready for book_pricer_models.ipynb')